# Data Cleaning 02 -- FRED Weekly

## Input
`Data/Data_Collection/Initial/02_FRED/fred_weekly.parquet`

## Purpose
Cleans weekly US macro/financial data from the FRED API. Keyed on `date` only (no PERMNO). Contains Fed balance sheet data (total assets, TGA, reserves), banking data (commercial bank credit, C&I loans), and jobless claims (initial and continued, first-release vintage). Key concerns addressed: mixed reporting days across series (Fed balance sheet on Wednesday, jobless claims on Saturday), structural NaN patterns from the mixed-day calendar, monthly series misplaced in the weekly file, and date range alignment with the rest of the pipeline.

## Stage 0: Load & Inspect
Basic shape, date range, column inventory, and dtype verification.

## Stage 1: Missing Data Audit
- Total NaN count and percentage across all factor cells
- Per-column NaN counts with first/last valid dates
- Per-row NaN distribution

## Stage 2: Weekly Structure Checks
- **Day-of-week distribution:** identifies which days of the week have observations, reflecting the different reporting schedules of each FRED series
- **Date gap distribution:** reports the frequency of inter-observation gaps in days
- **Per-series reporting frequency:** for each column, reports observation count, modal reporting day, average/median/max gap in days. This is how the monthly series (`consumer_loans`, `real_estate_loans`) were identified as misplaced -- they showed avg gap of 30 days instead of 7
- **Duplicate date check**
- **Long gap analysis:** flags any per-series gaps exceeding 14 days and prints the top 3 longest for investigation
- **Value range sanity checks:** verifies key series fall within expected ranges (e.g., Fed total assets $500B--$10T, initial claims 150K--10M, bank credit $5T--$25T)

## Stage 4: Clean & Save

### Columns Dropped (2)
- `consumer_loans` -- monthly series (264 obs, avg gap 30 days, reports first Friday of each month) misplaced in the weekly file during collection. 89% NaN because the DataFrame has rows for every reporting day across all series.
- `real_estate_loans` -- same issue, monthly reporting disguised as weekly, 89% NaN.

### Date Range Trimmed to 2004-01-01
Consistent with the rest of the pipeline.

### Forward-Fill Per Column (No Limit)
The 53--54% NaN rate before filling is entirely structural: each row corresponds to a date where at least one series reported, but the others did not (e.g., Fed balance sheet reports Wednesday, jobless claims report Saturday). Forward-fill per column carries each series' last reported value to the next reporting date. No limit is needed because all 7 remaining series report every single week with no skipped periods across the full sample (avg gap = 7 days, median = 7, max = 7 for all).

### Backfill (Limit 1)
Applied to handle the very first row per series if it is NaN (the single leading observation before the first report).

### Units Note
`initial_claims` and `continued_claims` are in persons (not thousands). FRED series ICSA/CCSA report raw counts (e.g., 211,000 not 211). This is correct and consistent with the FRED API output. No transformation applied.

### Note for Merge Pipeline
This data will be forward-filled from weekly to daily frequency during the merge stage. `net_liquidity = fed_assets - tga` can be computed there.

## Output
`Data/Data_Collection/Cleaned/02_FRED/fred_weekly_clean.parquet` -- 7 factor columns (down from 9 before cleaning)

In [1]:
# %% [markdown]
# # Data Cleaning: fred_weekly.parquet
#
# Source: Data/Data_Collection/Initial/02_FRED/fred_weekly.parquet
# Output: Data/Data_Collection/Cleaned/02_FRED/fred_weekly_clean.parquet
#
# Weekly US macro/financial data from the FRED API. No PERMNO — keyed on date only.
# Contains:
#   - Fed balance sheet: total assets, TGA, reserves (standard pull)
#   - Banking: commercial bank credit, C&I loans, consumer loans, real estate loans
#   - Jobless claims: initial and continued (first-release vintage)
#
# Key concerns:
#   - What day of the week are observations on? (Thursday for claims, Wednesday for Fed)
#   - Are there gaps in the weekly series?
#   - This data will be forward-filled to daily in the merge pipeline
#   - The collection code starts from 2003-01-01, not 2004

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../../../Data/Data_Collection/Initial/02_FRED/fred_weekly.parquet')
OUT_DIR  = Path('../../../Data/Data_Collection/Cleaned/02_FRED')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — fred_weekly")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique dates: {df['date'].nunique():,}")

factor_cols = [c for c in df.columns if c != 'date']

print(f"\nFactor columns ({len(factor_cols)}):")
for i, c in enumerate(factor_cols, 1):
    print(f"  {i:>3d}. {c:<25s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (10 rows) ---")
print(df.head(10).to_string(index=False))

print(f"\n--- Tail (10 rows) ---")
print(df.tail(10).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT — fred_weekly")
print("=" * 90)

n_rows = len(df)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_cells = n_rows * len(factor_cols)
total_nan = df[factor_cols].isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN ───────────────────────────────────────────────────────────
print(f"\n--- Per-Column NaN ---")
print(f"\n  {'Column':<25s} {'NaN %':>8s}  {'Count':>6s}  {'First Valid':>12s}  {'Last Valid':>12s}")
print("  " + "-" * 75)
for col in factor_cols:
    n = df[col].isna().sum()
    pct = n / n_rows * 100
    valid = df[df[col].notna()]['date']
    first = valid.min().date() if len(valid) > 0 else 'N/A'
    last = valid.max().date() if len(valid) > 0 else 'N/A'
    flag = " ← DROP" if pct >= 30 else (" ← INVESTIGATE" if pct >= 10 else "")
    print(f"  {col:<25s} {pct:>7.2f}%  {n:>6d}  {str(first):>12s}  {str(last):>12s}{flag}")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df[factor_cols].isna().sum(axis=1)
print(f"\n--- Per-Row NaN Distribution ---")
print(f"  Rows with 0 NaN: {(row_nan == 0).sum():>6,d} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with 1-3 NaN: {((row_nan >= 1) & (row_nan <= 3)).sum():>6,d}")
print(f"  Rows with >3 NaN: {(row_nan > 3).sum():>6,d}")
print(f"  Max NaN in any row: {row_nan.max()} out of {len(factor_cols)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: WEEKLY STRUCTURE CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: WEEKLY STRUCTURE CHECKS")
print("=" * 90)

# ── 2a. Day-of-week distribution ────────────────────────────────────────────
# Different FRED weekly series report on different days:
#   - Fed balance sheet (WALCL, WTREGEN, WRESBAL): Wednesday
#   - Jobless claims (ICSA, CCSA): Thursday (for the prior week)
#   - Banking data: varies
print(f"\n--- Day-of-week distribution ---")
df['_dow'] = df['date'].dt.dayofweek
df['_day_name'] = df['date'].dt.day_name()
dow_counts = df['_day_name'].value_counts().sort_index()
print(dow_counts.to_string())

# ── 2b. Date gap analysis ───────────────────────────────────────────────────
print(f"\n--- Date gap distribution ---")
date_diffs = df['date'].diff().dt.days.dropna()
print(f"  Gap distribution (days):")
for gap, count in date_diffs.value_counts().sort_index().items():
    pct = count / len(date_diffs) * 100
    print(f"    {int(gap):>3d} days: {count:>5,d} ({pct:.1f}%)")

# ── 2c. Per-series reporting frequency ───────────────────────────────────────
print(f"\n--- Per-series reporting frequency ---")
for col in factor_cols:
    valid = df[df[col].notna()].copy()
    if len(valid) < 2:
        print(f"  {col:<25s} {len(valid)} valid obs — INSUFFICIENT DATA")
        continue

    gaps = valid['date'].diff().dt.days.dropna()
    avg_gap = gaps.mean()
    median_gap = gaps.median()
    max_gap = gaps.max()
    n_obs = len(valid)

    # What day of week does this series report on?
    reporting_day = valid['date'].dt.day_name().mode().iloc[0]

    print(f"  {col:<25s} {n_obs:>5d} obs | day: {reporting_day:<10s} | "
          f"avg gap: {avg_gap:.0f}d | median: {median_gap:.0f}d | max: {max_gap:.0f}d")

# ── 2d. Check for duplicate dates ───────────────────────────────────────────
print(f"\n--- Duplicate dates ---")
n_dupes = df['date'].duplicated().sum()
if n_dupes == 0:
    print(f"  ✓ No duplicate dates")
else:
    print(f"  ⚠ {n_dupes} duplicate dates found")
    print(df[df['date'].duplicated(keep=False)].sort_values('date').head(10).to_string(index=False))

# ── 2e. Gap analysis per series (long gaps) ─────────────────────────────────
print(f"\n--- Long gaps (>14 days) per series ---")
for col in factor_cols:
    valid = df[df[col].notna()]['date'].sort_values()
    if len(valid) < 2:
        continue
    gaps = valid.diff().dt.days.dropna()
    long_gaps = gaps[gaps > 14]
    if len(long_gaps) > 0:
        print(f"\n  {col}: {len(long_gaps)} gaps > 14 days")
        # Show top 3 longest
        for idx in long_gaps.nlargest(3).index:
            gap_end = valid.loc[idx]
            gap_start = valid.loc[idx - 1] if (idx - 1) in valid.index else valid.iloc[valid.index.get_loc(idx) - 1]
            print(f"    {gap_start.date()} → {gap_end.date()} ({int(gaps.loc[idx])} days)")

# ── 2f. Value range sanity checks ───────────────────────────────────────────
print(f"\n--- Value range sanity checks ---")
checks = {
    'fed_assets':      (500000, 10000000, 'Fed total assets $M (expect ~$800B-$9T)'),
    'tga':             (0, 2000000,       'Treasury General Account $M'),
    'reserves':        (0, 5000000,       'Reserve balances $M'),
    'initial_claims':  (150, 10000,       'Initial claims (thousands)'),
    'continued_claims':(500, 30000,       'Continued claims (thousands)'),
    'bank_credit':     (5000, 25000,      'Bank credit $B'),
}

for col, (low, high, desc) in checks.items():
    if col not in df.columns:
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    n_out = ((vals < low) | (vals > high)).sum()
    if n_out > 0:
        print(f"  ⚠ {col:<25s} {n_out:>3d} values outside [{low:,}, {high:,}] — {desc}")
        print(f"    Actual range: {vals.min():,.0f} to {vals.max():,.0f}")
    else:
        print(f"  ✓ {col:<25s} all in [{low:,}, {high:,}]")

# Clean up temp columns
df = df.drop(columns=['_dow', '_day_name'])

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: SUMMARY — DECISIONS NEEDED BEFORE CLEANING")
print("=" * 90)

print(f"""
Review the output above and decide:

1. COLUMNS TO DROP:
   - Any series with ≥30% NaN
   - Any completely empty series

2. DATE RANGE:
   - Raw file starts 2003-01-01. Trim to 2004-01-01?
   - Or keep 2003 since weekly data has no derived lookback?

3. MIXED REPORTING DAYS:
   - Different series may report on different days of the week
   - When forward-filling to daily in the merge pipeline, this means
     some factors update on Wednesday, others on Thursday
   - This is correct behaviour — no action needed here

4. NaN HANDLING:
   - The weekly file has a different NaN pattern than daily:
     each series only has values on its reporting day, so most
     rows will be NaN for most columns (since rows span all dates
     where ANY series has a value)
   - Forward-fill per column is the correct approach

5. NOTE FOR MERGE PIPELINE:
   - This data will be forward-filled to daily frequency during merge
   - net_liquidity = fed_assets - tga - rrp should be computed there
     (but rrp was dropped from fred_daily — revisit if needed)

Paste back the output and I will write the cleaning cell.
""")

STAGE 0: LOAD & INSPECT — fred_weekly

Shape: 2,485 rows × 10 columns
Date range: 2003-01-01 → 2024-12-28
Unique dates: 2,485

Factor columns (9):
    1. fed_assets                float64        
    2. tga                       float64        
    3. reserves                  float64        
    4. bank_credit               float64        
    5. ci_loans                  float64        
    6. consumer_loans            float64        
    7. real_estate_loans         float64        
    8. initial_claims            float64        
    9. continued_claims          float64        

--- Head (10 rows) ---
      date  fed_assets    tga  reserves  bank_credit  ci_loans  consumer_loans  real_estate_loans  initial_claims  continued_claims
2003-01-01    730994.0 5016.0   10667.0    5465.9682  953.0994        610.4253          2036.2306             NaN               NaN
2003-01-04         NaN    NaN       NaN          NaN       NaN             NaN                NaN        393000.0         34

In [2]:
# %% [markdown]
# ## Stage 4: Clean & Save
#
# **Cleaning decisions and rationale:**
#
# **Columns dropped (2):**
# - `consumer_loans` — monthly series (264 obs, avg gap 30 days, reports first Friday
#   of each month). Misplaced in the weekly file during collection. Unusable at weekly
#   frequency — 89% NaN because the DataFrame has rows for every reporting day across
#   all series, and this one only updates monthly.
# - `real_estate_loans` — same issue. Monthly reporting disguised as weekly. 89% NaN.
#
# **No columns have genuine missing data:** all 7 remaining series show avg gap = 7 days,
# median gap = 7 days, max gap = 7 days. Every single week is accounted for with zero
# skipped reporting periods across the full 2003–2024 sample.
#
# **Date range trimmed to 2004-01-01:** consistent with the rest of the pipeline.
#
# **Forward-fill per column:** the 53–54% NaN rate is entirely structural — each row
# corresponds to a date where at least one series reported, but the others didn't
# (Fed balance sheet reports Wednesday, jobless claims report Saturday). Forward-fill
# per column carries each series' last reported value to the next reporting date.
# No limit needed because there are no genuine gaps — every series reports every week.
#
# **Units note:** `initial_claims` and `continued_claims` are in persons (not thousands).
# FRED series ICSA/CCSA report raw counts (e.g., 211,000 not 211). This is correct
# and consistent with the FRED API output. No transformation applied — normalisation
# happens in the model pipeline.
#
# **For the merge pipeline:** this data will be forward-filled from weekly to daily
# frequency. `net_liquidity = fed_assets - tga` can be computed there.
#
# **Factors retained: 7** (was 9 before cleaning)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 4: CLEAN & SAVE")
print("=" * 90)

# ── 4a. Drop monthly series ─────────────────────────────────────────────────
drop_cols = ['consumer_loans', 'real_estate_loans']
drop_cols_present = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols_present)

factor_cols = [c for c in df.columns if c != 'date']
print(f"\n  Dropped {len(drop_cols_present)} columns: {drop_cols_present}")
print(f"  Remaining: {len(factor_cols)} factor columns")

# ── 4b. Trim to 2004-01-01 ──────────────────────────────────────────────────
n_before = len(df)
df = df[df['date'] >= '2004-01-01'].reset_index(drop=True)
n_trimmed = n_before - len(df)
print(f"  Trimmed {n_trimmed} rows before 2004-01-01 ({n_before} → {len(df)})")

# ── 4c. Forward-fill per column ──────────────────────────────────────────────
nan_before = df[factor_cols].isna().sum().sum()
df = df.sort_values('date')
df[factor_cols] = df[factor_cols].ffill()
nan_after = df[factor_cols].isna().sum().sum()
print(f"\n  Forward-fill (no limit): {nan_before:,} → {nan_after:,} NaN")

# ── 4d. Handle any remaining leading NaN (first row per series) ─────────────
if nan_after > 0:
    # Backfill limit=1 catches the very first row if it's NaN
    df[factor_cols] = df[factor_cols].bfill(limit=1)
    nan_final = df[factor_cols].isna().sum().sum()
    print(f"  Backfill (limit=1): {nan_after:,} → {nan_final:,} NaN")
else:
    nan_final = 0

# ── 4e. Final NaN check ─────────────────────────────────────────────────────
nan_check = df[factor_cols].isna().sum()
nan_cols = nan_check[nan_check > 0]

if len(nan_cols) == 0:
    print(f"\n  ✓ Zero NaN — all clean")
else:
    print(f"\n  Remaining NaN ({len(nan_cols)} columns):")
    for col, n in nan_cols.items():
        print(f"    {col}: {n}")

# ── 4f. Drop rows that are entirely NaN across all factors ───────────────────
# After forward-fill, some rows may still be redundant (e.g., a Saturday row
# now has the same values as the preceding Wednesday). We keep all rows here
# since the merge pipeline will align to trading days anyway.

# ── 4g. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Factor columns: {len(factor_cols)}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

print(f"\n  Factor list ({len(factor_cols)} columns):")
for i, c in enumerate(factor_cols, 1):
    print(f"    {i:>3d}. {c}")

print(f"\n  Sample (first 5 rows):")
print(df.head(5).to_string(index=False))
print(f"\n  Sample (last 5 rows):")
print(df.tail(5).to_string(index=False))

# ── 4h. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'fred_weekly_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nCleaning complete.")

STAGE 4: CLEAN & SAVE

  Dropped 2 columns: ['consumer_loans', 'real_estate_loans']
  Remaining: 7 factor columns
  Trimmed 112 rows before 2004-01-01 (2485 → 2373)

  Forward-fill (no limit): 8,944 → 12 NaN
  Backfill (limit=1): 12 → 5 NaN

  Remaining NaN (5 columns):
    fed_assets: 1
    tga: 1
    reserves: 1
    bank_credit: 1
    ci_loans: 1

  Final shape: 2,373 rows × 8 columns
  Factor columns: 7
  Date range: 2004-01-01 → 2024-12-28

  Factor list (7 columns):
      1. fed_assets
      2. tga
      3. reserves
      4. bank_credit
      5. ci_loans
      6. initial_claims
      7. continued_claims

  Sample (first 5 rows):
      date  fed_assets    tga  reserves  bank_credit  ci_loans  initial_claims  continued_claims
2004-01-01         NaN    NaN       NaN          NaN       NaN        356000.0         3188000.0
2004-01-03    755829.0 5319.0    9839.0    5805.2350  878.5229        356000.0         3188000.0
2004-01-07    755829.0 5319.0    9839.0    5805.2350  878.5229     